# ForecastLLM - Week 6 Day 2: Feature Engineering for Time Series

This week - build a model that predicts future values from historical data.

A model that can estimate the next value in a time series, from recent observations.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Baselines and Evaluation  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll transform a cleaned series into supervised features.
Lag-based features are a strong and simple starting point.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Feature Engineering</h2>
            <span style="color:#181;">Feature engineering translates raw sequential data into model-ready inputs.
            For many forecasting problems, simple lag features provide a fast baseline and often strong practical value.</span>
        </td>
    </tr>
</table>


In [6]:
import os
from importlib import import_module

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# replaced pricing/text pipeline with local time-series loading
from week6.data_loader import load_sample_series


# The next cell is where we select data source

Default behavior is local M4 hourly data via `FORECAST_DATA_PATH` (or the loader default path).

Synthetic data remains available only as explicit fallback (`USE_SYNTHETIC_FALLBACK=True`).


In [7]:
load_dotenv()

USE_SYNTHETIC_FALLBACK = False
MIN_ROWS = 240

print(f"FORECAST_DATA_PATH={os.getenv('FORECAST_DATA_PATH')}")
print(f"USE_SYNTHETIC_FALLBACK={USE_SYNTHETIC_FALLBACK}")

FORECAST_DATA_PATH=/home/geo/Projects/Python/forecastllm/data/m4/processed/hourly_longest_series.csv
USE_FALLBACK=False


In [8]:
# Load one M4 hourly series (synthetic fallback is explicit only)

data_loader = import_module("week6.data_loader")
data_loader = import_module("importlib").reload(data_loader)

try:
    ts_df = data_loader.load_sample_series()
    source_label = os.getenv("FORECAST_DATA_PATH") or "default_m4_processed"
except Exception as e:
    if not USE_SYNTHETIC_FALLBACK:
        raise
    ts_df = data_loader.load_synthetic_series(periods=240)
    source_label = "synthetic_fallback"
    print(f"Loader failed ({e}); using explicit synthetic fallback")

if not isinstance(ts_df, pd.DataFrame):
    raise TypeError(f"load_sample_series must return a pandas DataFrame, got {type(ts_df)}")
if not {"timestamp", "value"}.issubset(ts_df.columns):
    raise ValueError("Loaded dataframe must include 'timestamp' and 'value' columns")

ts_df = ts_df[["timestamp", "value"]].copy()
ts_df["value"] = pd.to_numeric(ts_df["value"], errors="coerce")
ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], errors="coerce")
ts_df = ts_df.dropna(subset=["timestamp", "value"]).sort_values("timestamp").reset_index(drop=True)

if len(ts_df) < MIN_ROWS:
    raise RuntimeError(f"Need at least {MIN_ROWS} rows, got {len(ts_df)}")

print(f"Loaded {len(ts_df):,} observations from {source_label}")
print(ts_df.head(2))

Loaded 960 observations from /home/geo/Projects/Python/forecastllm/data/m4/processed/hourly_longest_series.csv
            timestamp  value
0 2000-01-01 00:00:00   12.8
1 2000-01-01 01:00:00   12.1


In [9]:
ts_df.head(5)

,timestamp,value
0,2000-01-01 00:00:00,12.8
1,2000-01-01 01:00:00,12.1
2,2000-01-01 02:00:00,11.6
3,2000-01-01 03:00:00,11.2
4,2000-01-01 04:00:00,10.8


In [10]:
# Give every row an id for easier inspection

ts_df["row_id"] = np.arange(len(ts_df))
ts_df[["row_id", "timestamp", "value"]].head(3)


,row_id,timestamp,value
0,0,2000-01-01 00:00:00,12.8
1,1,2000-01-01 01:00:00,12.1
2,2,2000-01-01 02:00:00,11.6


In [11]:
# Lag feature configuration for hourly M4
LAG_COLUMNS = ["lag_1", "lag_2", "lag_3", "lag_7", "lag_24"]

In [12]:
print(ts_df[["timestamp", "value"]].head(10).to_string(index=False))

          timestamp  value
2000-01-01 00:00:00   12.8
2000-01-01 01:00:00   12.1
2000-01-01 02:00:00   11.6
2000-01-01 03:00:00   11.2
2000-01-01 04:00:00   10.8
2000-01-01 05:00:00   10.4
2000-01-01 06:00:00   10.3
2000-01-01 07:00:00   11.4
2000-01-01 08:00:00   13.1
2000-01-01 09:00:00   14.6


In [13]:
# Build lag features for hourly data
# lag_1/lag_2/lag_3/lag_7 = short-memory lags
# lag_24 = daily seasonal lag (hourly cadence)

df = ts_df[["timestamp", "value"]].copy()
df["lag_1"] = df["value"].shift(1)
df["lag_2"] = df["value"].shift(2)
df["lag_3"] = df["value"].shift(3)
df["lag_7"] = df["value"].shift(7)
df["lag_24"] = df["value"].shift(24)

# target is the value at the current row, using prior rows as predictors
df["target"] = df["value"]

df.head(10)

,timestamp,value,lag_1,lag_2,lag_3,lag_7,target
0,2000-01-01 00:00:00,12.8,NaN,NaN,NaN,NaN,12.8
1,2000-01-01 01:00:00,12.1,12.8,NaN,NaN,NaN,12.1
2,2000-01-01 02:00:00,11.6,12.1,12.8,NaN,NaN,11.6
3,2000-01-01 03:00:00,11.2,11.6,12.1,12.8,NaN,11.2
4,2000-01-01 04:00:00,10.8,11.2,11.6,12.1,NaN,10.8
5,2000-01-01 05:00:00,10.4,10.8,11.2,11.6,NaN,10.4
6,2000-01-01 06:00:00,10.3,10.4,10.8,11.2,NaN,10.3
7,2000-01-01 07:00:00,11.4,10.3,10.4,10.8,12.8,11.4
8,2000-01-01 08:00:00,13.1,11.4,10.3,10.4,12.1,13.1
9,2000-01-01 09:00:00,14.6,13.1,11.4,10.3,11.6,14.6


In [14]:
# Inspect NaNs introduced by lagging
df.isna().sum()

timestamp    0
value        0
lag_1        1
lag_2        2
lag_3        3
lag_7        7
target       0
dtype: int64

In [15]:
FEATURE_COLUMNS = LAG_COLUMNS
TARGET_COLUMN = "target"

In [16]:
def make_supervised(frame, feature_columns, target_column="target"):
    out = frame.copy()
    out = out.dropna(subset=feature_columns + [target_column]).copy()
    return out


In [17]:
df.head(1)

,timestamp,value,lag_1,lag_2,lag_3,lag_7,target
0,2000-01-01,12.8,NaN,NaN,NaN,NaN,12.8


In [18]:
make_supervised(df, FEATURE_COLUMNS, TARGET_COLUMN).head()

,timestamp,value,lag_1,lag_2,lag_3,lag_7,target
7,2000-01-01 07:00:00,11.4,10.3,10.4,10.8,12.8,11.4
8,2000-01-01 08:00:00,13.1,11.4,10.3,10.4,12.1,13.1
9,2000-01-01 09:00:00,14.6,13.1,11.4,10.3,11.6,14.6
10,2000-01-01 10:00:00,16.0,14.6,13.1,11.4,11.2,16.0
11,2000-01-01 11:00:00,17.2,16.0,14.6,13.1,10.8,17.2


In [19]:
def chronological_split(frame, train_frac=0.8):
    cutoff_idx = int(len(frame) * train_frac)
    train = frame.iloc[:cutoff_idx].copy()
    test = frame.iloc[cutoff_idx:].copy()
    return train, test


In [20]:
supervised_df = make_supervised(df, FEATURE_COLUMNS, TARGET_COLUMN)
supervised_df.shape

(953, 7)

In [21]:
# Calendar features used throughout week6/week7
supervised_df["day_of_week"] = supervised_df["timestamp"].dt.dayofweek
supervised_df["month"] = supervised_df["timestamp"].dt.month

In [22]:
# Feature matrix and target vector
MODEL_FEATURES = FEATURE_COLUMNS + ["day_of_week", "month"]
X = supervised_df[MODEL_FEATURES].copy()
y = supervised_df[TARGET_COLUMN].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (953, 6)
y shape: (953,)


In [23]:
# Chronological split (no shuffle)
split_idx = int(len(supervised_df) * 0.8)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print("Split index:", split_idx)


Split index: 762


In [24]:
(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

((762, 6), (191, 6), (762,), (191,))

In [25]:
# Quick safety checks before modeling
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert X_train.index.max() < X_test.index.min()
print("Chronological split checks passed")


Chronological split checks passed


In [26]:
print(supervised_df[["timestamp"] + MODEL_FEATURES + [TARGET_COLUMN]].head(10).to_string(index=False))

          timestamp  lag_1  lag_2  lag_3  lag_7  day_of_week  month  target
2000-01-01 07:00:00   10.3   10.4   10.8   12.8            5      1    11.4
2000-01-01 08:00:00   11.4   10.3   10.4   12.1            5      1    13.1
2000-01-01 09:00:00   13.1   11.4   10.3   11.6            5      1    14.6
2000-01-01 10:00:00   14.6   13.1   11.4   11.2            5      1    16.0
2000-01-01 11:00:00   16.0   14.6   13.1   10.8            5      1    17.2
2000-01-01 12:00:00   17.2   16.0   14.6   10.4            5      1    18.0
2000-01-01 13:00:00   18.0   17.2   16.0   10.3            5      1    18.7
2000-01-01 14:00:00   18.7   18.0   17.2   11.4            5      1    19.1
2000-01-01 15:00:00   19.1   18.7   18.0   13.1            5      1    19.3
2000-01-01 16:00:00   19.3   19.1   18.7   14.6            5      1    19.2


In [27]:
print(X_train.head(5))

    lag_1  lag_2  lag_3  lag_7  day_of_week  month
7    10.3   10.4   10.8   12.8            5      1
8    11.4   10.3   10.4   12.1            5      1
9    13.1   11.4   10.3   11.6            5      1
10   14.6   13.1   11.4   11.2            5      1
11   16.0   14.6   13.1   10.8            5      1


In [28]:
print(y_train.head(5))

7     11.4
8     13.1
9     14.6
10    16.0
11    17.2
Name: target, dtype: float64


## I've put exactly this logic into helper functions

- Builds lag features from one cleaned time series
- Drops lag-related NaNs
- Creates chronological train/test split without shuffling

This gives us a reliable Day 2 baseline for forecasting experiments.


In [ ]:
# earlier duplicate helper execution removed; keeping the later version that adds calendar features

In [29]:
# run helper-based pipeline end-to-end
supervised_df = make_supervised(df, FEATURE_COLUMNS, TARGET_COLUMN)
supervised_df["day_of_week"] = supervised_df["timestamp"].dt.dayofweek
supervised_df["month"] = supervised_df["timestamp"].dt.month
train_df, test_df = chronological_split(supervised_df, train_frac=0.8)

print(f"Supervised rows: {len(supervised_df):,}")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


Supervised rows: 953
Train rows: 762
Test rows: 191


In [30]:
# final sanity check for missing values
print("NaNs in X_train:", int(X_train.isna().sum().sum()))
print("NaNs in X_test:", int(X_test.isna().sum().sum()))


NaNs in X_train: 0
NaNs in X_test: 0


In [31]:
# inspect the last prepared row
supervised_df.tail(1)

,timestamp,value,lag_1,lag_2,lag_3,lag_7,target,day_of_week,month
959,2000-02-09 23:00:00,19.8,20.5,21.6,23.2,26.3,19.8,2,2


In [32]:
# keep ordering explicit for forecasting (no shuffle)
print("Train time range:", train_df["timestamp"].min(), "->", train_df["timestamp"].max())
print("Test time range:", test_df["timestamp"].min(), "->", test_df["timestamp"].max())


Train time range: 2000-01-01 07:00:00 -> 2000-02-02 00:00:00
Test time range: 2000-02-02 01:00:00 -> 2000-02-09 23:00:00


In [33]:
# inspect one training example
idx = X_train.index[0]
print("Features:")
print(X_train.loc[idx])
print("Target:", y_train.loc[idx])


Features:
lag_1          10.3
lag_2          10.4
lag_3          10.8
lag_7          12.8
day_of_week     5.0
month           1.0
Name: 7, dtype: float64
Target: 11.4


In [34]:
# remove fields that we don't need for immediate model input
model_ready_train = train_df[MODEL_FEATURES + [TARGET_COLUMN]].copy()
model_ready_test = test_df[MODEL_FEATURES + [TARGET_COLUMN]].copy()

model_ready_train.head(2)


,lag_1,lag_2,lag_3,lag_7,day_of_week,month,target
7,10.3,10.4,10.8,12.8,5,1,11.4
8,11.4,10.3,10.4,12.1,5,1,13.1


## Persist step (deferred)

In the original notebook, Day 2 pushes pre-processed data artifacts externally.

For ForecastLLM Day 2, we keep artifacts local and move directly to Day 3 baselines.


In [35]:
# local artifacts for Day 3
artifacts = {
    "features": MODEL_FEATURES,
    "target": TARGET_COLUMN,
    "split": {"train_rows": len(model_ready_train), "test_rows": len(model_ready_test)},
}

print(artifacts)


{'features': ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'day_of_week', 'month'], 'target': 'target', 'split': {'train_rows': 762, 'test_rows': 191}}


## Prepared outputs

- `X_train`, `X_test`, `y_train`, `y_test`
- `model_ready_train`, `model_ready_test`

These are ready for baseline models and MAE/sMAPE evaluation in Day 3.
